In [ ]:
# Needed imports
import torch

import pandas as pd
import unicodedata as ud

from nltk import word_tokenize
from torch import nn

In [ ]:
# LSTM Classifier Model Definition
class LSTMModel(nn.Module):
    def __init__(self, device, input_size, hidden_size, vocab_size, no_lstm_layers=1):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.device = device
        self.embedding = nn.Embedding(vocab_size, input_size)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=no_lstm_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        h0, c0 = (torch.zeros(self.num_layers, x.shape[0], self.hidden_size).to(self.device),
                torch.zeros(self.num_layers, x.shape[0], self.hidden_size).to(self.device))
        x = self.embedding(x)
        x, (h0, c0) = self.lstm(x, (h0, c0))
        x = x.contiguous().view(-1, self.hidden_size)
        x = self.fc(x[:, -1, :])
        self.fc_weights = x
        return x

In [ ]:
# Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"], engine='fastparquet')
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"], engine='fastparquet')
print(f"Train size: {len(df_train)}, Validation size: {len(df_val)}")

# Test sets
df_train_ar = df_train[df_train['lang'] == 'ar']
df_train_ko = df_train[df_train['lang'] == 'ko']
df_train_te = df_train[df_train['lang'] == 'te']

# Validation sets
df_val_ar = df_val[df_val['lang'] == 'ar']
df_val_ko = df_val[df_val['lang'] == 'ko']
df_val_te = df_val[df_val['lang'] == 'te']

def tokenize_question(question):
    # Remove all punctuation characters, keeping in mind that arabic is written from right to left
    question = ''.join([char for char in question if not ud.category(char).startswith('P')])
    # Tokenize the question into words
    words = word_tokenize(question)
    return words